## Reference diff: GSE202051 (Hwang et al. 2022) vs. Peng et al. Run D

### Aim
* Compare the newly built GSE202051 pseudo-bulk reference against this project's existing Peng et al. "Run D" reference, to assess whether it is a credible alternative for the project's long-running dead/ambiguous-profile problem.

### Input
* `data/deconvolution/xdec/paper_style_ref/refs_pseudobulk_raw.tsv` + `ref_metadata_pseudobulk_raw.tsv` — Peng et al. Run D (this project's existing, methodologically-sound Peng reference: raw untransformed pseudobulk, XDec does the quantile + Equation-1 transform).
* `data/deconvolution/xdec/gse202051_level1_pseudobulk/refs.tsv` + `ref_metadata.tsv` — new reference built from GSE202051 (Hwang et al. 2022, *Nat Genet*, PMID 35902743), 43-patient snRNA-seq PDAC atlas, via the XDec-SM Workbench's `prepare` command (`Level 1 Annotation`, donor/sample-aware pseudobulking, `GSE202051_totaldata-final-toshare.h5ad.gz`).

### Output
* Dimension / gene-overlap / pseudobulking-methodology comparison
* Cell-type taxonomy correspondence, classified gained / lost / equivalent / unresolved
* Cross-reference to notebook 11's prior finding on malignant/non-malignant ductal splitting

### Steps
* Load both reference matrices and metadata tables
* Compute gene overlap, pseudobulk counts per cell type, value-type (integer vs. pre-normalized) check
* Tabulate the cell-type taxonomy correspondence by hand (requires biological judgment, not a programmatic join)
* Revisit notebook 11's finding on whether supplying a malignant/non-malignant split alone fixed the dead-profile problem

### Result
* GSE202051 covers more genes (22,164 vs. 17,004) with 95% of Peng's gene list recovered, but has far fewer pseudobulks (314 vs. 909) because each one aggregates whole donor samples rather than 5-cell technical chunks.
* Taxonomy: malignant vs. non-malignant epithelial is newly resolved; stroma/vasculature splits further (CAF/Pericyte/Vascular smooth muscle); immune resolution is coarser (Lymphoid only, vs. Peng's T/B split).
* Notebook 11 already tested supplying the malignant/non-malignant split on Peng's own data and found it did **not** fix the dead-profile problem (in fact dead-profile dominance rose from 66% to 76-77%) — so the taxonomy fix alone is not expected to resolve the artifact; GSE202051's value as a reference rests on its different cohort, assay, and genuine donor-level replicate structure, not the taxonomy change by itself.

### Conclusion
* GSE202051 is a structurally different reference (true biological replicates instead of arbitrary 5-cell pools, broader gene coverage, malignant/non-malignant epithelial split) but is not a strict superset of Peng's taxonomy (coarser immune resolution) and is not expected, on its own, to resolve the dead-profile artifact given notebook 11's prior result. Actually running Stage 0/1 against the n=150 bulk cohort with this reference is the direct empirical test of whether it helps, and is the natural next step beyond this notebook.


### Background

* **Why this reference exists**: searching for an alternative scRNA/snRNA PDAC reference (via the XDec-SM Workbench's GEO discovery tool) to test against the project's long-running dead/ambiguous-profile artifact, independent of the k-selection and taxonomy-splitting angles already exhausted on Peng et al.'s own data (see [[project-xdec-replication-status]], notebook 11).
* **Two real bugs found and fixed while building it** (in `tools/XDec_SM/python/xdec_auto.py` and `reference_io.py`, collaborator-bundle copy):
  1. Gzip-wrapped container files (`.h5ad.gz`, `.h5.gz`, `.loom.gz`) were invisible to the discovery/validation pipeline — classifier and suffix-detection regexes only matched the un-gzipped extension.
  2. `prepare()`'s barcode-overlap check rebuilt a 225k-element Python `set()` on every iteration of a 225k-row loop (O(n²)) — would have run for hours; fixed by hoisting the `set()` out of the loop.
* **How the new reference was built**: GSE202051's full combined object (`GSE202051_totaldata-final-toshare.h5ad.gz`, 43 patients / 44 samples, 224,988 cells) was used with `--type-column "Level 1 Annotation"` (12 categories) and `--stratify-columns sampleid --pseudobulk-mode donor_aware`, i.e. one pseudobulk per (cell type, sample) pair, summing all QC-passing cells (`--min-counts 500 --min-genes 200 --min-cells-per-type 20`, defaults). Result: 314 pseudobulks across 11 of the 12 types (`Intra-pancreatic neurons` dropped — only 25 cells total in the whole dataset).


In [1]:
from pathlib import Path
import os

import numpy as np
import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "tools/XDec_SM/python/xdec.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start from progress/ or one of its subdirectories.")
os.chdir(ROOT)

PENG_REFS = "data/deconvolution/xdec/paper_style_ref/refs_pseudobulk_raw.tsv"
PENG_META = "data/deconvolution/xdec/paper_style_ref/ref_metadata_pseudobulk_raw.tsv"
NEW_REFS = "data/deconvolution/xdec/gse202051_level1_pseudobulk/refs.tsv"
NEW_META = "data/deconvolution/xdec/gse202051_level1_pseudobulk/ref_metadata.tsv"

peng_refs = pd.read_csv(PENG_REFS, sep="\t", index_col=0)
peng_meta = pd.read_csv(PENG_META, sep="\t", index_col=0)
new_refs = pd.read_csv(NEW_REFS, sep="\t", index_col=0)
new_meta = pd.read_csv(NEW_META, sep="\t", index_col=0, usecols=["Sample_ID", "Cell_Type", "n_cells"])

print(f"Peng Run D:  {peng_refs.shape[0]:,} genes x {peng_refs.shape[1]:,} pseudobulks")
print(f"GSE202051:   {new_refs.shape[0]:,} genes x {new_refs.shape[1]:,} pseudobulks")


Peng Run D:  17,004 genes x 909 pseudobulks
GSE202051:   22,164 genes x 314 pseudobulks


In [2]:
peng_genes = set(peng_refs.index)
new_genes = set(new_refs.index)
shared = peng_genes & new_genes

print(f"Shared genes:         {len(shared):,} ({len(shared) / len(peng_genes):.1%} of Peng's list, {len(shared) / len(new_genes):.1%} of GSE202051's list)")
print(f"Peng-only genes:      {len(peng_genes - new_genes):,}")
print(f"GSE202051-only genes: {len(new_genes - peng_genes):,}")


Shared genes:         16,191 (95.2% of Peng's list, 73.1% of GSE202051's list)
Peng-only genes:      813
GSE202051-only genes: 5,973


In [3]:
peng_is_int = np.allclose(peng_refs.to_numpy(), np.round(peng_refs.to_numpy()))
new_is_int = np.allclose(new_refs.to_numpy(), np.round(new_refs.to_numpy()))
print(f"Peng Run D values are all integer-valued:  {peng_is_int}  (cells were depth-normalized upstream before pseudobulking)")
print(f"GSE202051 values are all integer-valued:   {new_is_int}  (raw UMI counts summed per donor sample)")


Peng Run D values are all integer-valued:  False  (cells were depth-normalized upstream before pseudobulking)
GSE202051 values are all integer-valued:   True  (raw UMI counts summed per donor sample)


In [4]:
peng_counts = peng_meta["Cell_Type"].value_counts().rename("Peng Run D (n pseudobulks, fixed 5 cells each)")
new_counts = new_meta["Cell_Type"].value_counts().rename("GSE202051 (n pseudobulks, donor/sample-aware)")
pd.concat([peng_counts, new_counts], axis=1)

,"Peng Run D (n pseudobulks, fixed 5 cells each)","GSE202051 (n pseudobulks, donor/sample-aware)"
Cell_Type,,
T cell,100.0,NaN
blood vessel endothelial cell,100.0,NaN
enteroendocrine cell,100.0,NaN
fibroblast,100.0,NaN
lymphocyte of B lineage,100.0,NaN
myeloid leukocyte,100.0,NaN
pancreatic acinar cell,100.0,NaN
pancreatic ductal cell,100.0,NaN
pancreatic stellate cell,100.0,NaN


In [5]:
new_meta.groupby("Cell_Type")["n_cells"].agg(["count", "min", "median", "max"]).sort_values("count", ascending=False)

,count,min,median,max
Cell_Type,,,,
Endothelial,41,58,265.0,1454
Cancer-associated fibroblast,40,44,931.0,5430
Pericyte,39,27,139.0,1739
Epithelial (malignant),37,32,1019.0,7167
Myeloid,36,32,227.5,909
Endocrine,33,20,157.0,1640
Lymphoid,28,24,165.0,858
Epithelial (non-malignant),27,25,1065.0,3984
Schwann,20,20,37.0,471


### Cell-type taxonomy correspondence

Hand-mapped from each reference's published cell-type labels (not a programmatic join — the two taxonomies use different ontologies).

**✅ Direct equivalents**
* `myeloid leukocyte` ↔ Myeloid
* `blood vessel endothelial cell` ↔ Endothelial
* `enteroendocrine cell` ↔ Endocrine

**✅ Resolved — the motivating gap**
* Peng's single `pancreatic ductal cell` → split into **Epithelial (malignant)** + **Epithelial (non-malignant)**. This is the gap flagged throughout the investigation (epithelial compartment "never splits further at any k", notebook 08).

**⚠️ Gained resolution, different axis**
* Peng's `fibroblast` + `pancreatic stellate cell` (2 types) → **Cancer-associated fibroblast** + **Pericyte** + **Vascular smooth muscle** (3 types). Finer, but not 1:1 — stellate cell activation state doesn't map cleanly onto CAF vs. pericyte.

**⚠️ Lost resolution**
* Peng's `T cell` + `lymphocyte of B lineage` (2 types) → single **Lymphoid** bucket. A real regression in immune granularity.
* Peng's `pancreatic acinar cell` folded into GSE202051's **Epithelial (non-malignant)** rather than kept distinct.

**❓ Unresolved in both**
* Peng's `neural cell`: 9 cells total. GSE202051's `Intra-pancreatic neurons`: 25 cells total, dropped entirely from the build (table above). Both references are effectively blind to this compartment.

**🆕 New, no Peng counterpart**
* Adipocyte (6 pseudobulks), Schwann (20 pseudobulks).


### Cross-reference: notebook 11 already tested the malignant/non-malignant split

Before treating GSE202051's malignant/non-malignant epithelial split as progress, it's worth citing `xdec_autotuned_n150/11_reference_preprocessing_summary.ipynb`'s own test of exactly this split, built directly from **Peng's own `Cell_type` labels** (same cohort, same cells, no new data) at fixed k=13:

| Variant | What was split | Best malignant-ductal match (Spearman r) | Combined dead-profile dominance |
|---|---|---|---|
| A | nothing (`celltype1`, merged) | 0.44 | 66% |
| B | normal vs. malignant ductal (`Cell_type`) | 0.40 | 66% |
| C | B + paper-style pseudobulk | 0.64 | 76% |
| D | C + Classical-like vs. Basal-like within malignant ductal | 0.65 | 77% |

**The two metrics moved in opposite directions**: splitting the reference sharpened the single best correlation (0.44 → 0.65), but the fraction of the cohort stuck under a dead profile never improved and drifted upward (66% → 77%). Notebook 11's own conclusion: "the reference gained exactly the resolution needed ... and XDec's k=13 fit still collapsed both subtypes onto one profile, while the dead-profile burden kept growing rather than shrinking."

**Why GSE202051 is still worth testing despite this**: the prior test only varied the cell-type *label*, keeping everything else fixed (same patients, same single-cell assay, same technical 5-cell pseudobulking). GSE202051 differs on three axes notebook 11 never varied — a different cohort (43 Hwang et al. patients vs. Peng's), a different assay (snRNA-seq vs. scRNA-seq), and genuine donor/sample-level biological pseudobulking instead of arbitrary 5-cell technical chunks. Any of those could matter independently of the taxonomy change — but the taxonomy change by itself should not be expected to be what fixes it.


### Conclusion

* GSE202051 is a genuinely different reference, not a drop-in upgrade: it gains the malignant/non-malignant epithelial split and finer stroma/vasculature resolution, trades away some immune granularity (T/B lumped into Lymphoid), and switches from technical 5-cell pseudobulk chunks to real donor/sample-level biological replicates (314 pseudobulks across 44 samples vs. 909 across no biological structure at all).
* Notebook 11 already shows the taxonomy change alone (tested on Peng's own data) does not fix the dead-profile problem and may worsen it — so this reference's value, if any, has to come from its different cohort, assay, or donor-aware pseudobulking, not the malignant/non-malignant split by itself.
* **Next step**: run XDec Stage 0/1 against the n=150 TCGA-PAAD bulk cohort using this reference, and check whether dead-profile dominance and malignant-compartment correlation move together or apart from the Peng Run D baseline — the direct empirical test this notebook sets up but does not itself run.
